# 05_04 · Modelos Avanzados — SPARK TEC

Optimización del detector de anomalías eléctrico.

Sin etiquetas reales → evaluación mediante:
- Estabilidad del score entre máquinas
- Distribución y separación de anomalías detectadas
- Comparativa Isolation Forest vs One-Class SVM vs LOF

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

PROC_DIR  = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# Cargar datos agregados globales
df = pd.read_parquet(os.path.join(PROC_DIR, 'spark_tec_scored.parquet'))
print(f'Total registros: {len(df):,}')
print(f'Máquinas: {df["machine"].nunique()} — {sorted(df["machine"].unique())}')
print(f'Features disponibles: {[c for c in df.columns if c not in ("machine","anomaly_score","is_anomaly")]}')

## 1. Features y baseline

Features eléctricas: potencia, corriente por fase, THD, PF, frecuencia (36 features × {mean, std, max}).
Baseline: Isolation Forest con `contamination=0.05` (usado en 03_04).

In [ ]:
FEAT_COLS = [c for c in df.columns
             if c not in ('machine', 'anomaly_score', 'is_anomaly')]

# Muestra estratificada por máquina para velocidad (máx 50k por máquina)
sample = (df.groupby('machine', group_keys=False)
            .apply(lambda g: g.sample(min(len(g), 50_000), random_state=42)))
X_all = sample[FEAT_COLS].values
machines_all = sample['machine'].values

# Escalar
sc = StandardScaler()
X_sc = sc.fit_transform(X_all)

# Baseline IF
if_base = IsolationForest(n_estimators=100, contamination=0.05,
                          random_state=42, n_jobs=-1)
if_base.fit(X_sc)
scores_base = -if_base.score_samples(X_sc)
anomaly_rate_base = (if_base.predict(X_sc) == -1).mean()
print(f'Baseline IF contamination=0.05 → tasa anomalía: {anomaly_rate_base:.2%}')
print(f'Score: media={scores_base.mean():.3f} | std={scores_base.std():.3f}')

## 2. Barrido de contamination — Isolation Forest

In [ ]:
contaminations = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15]
cont_results = []

for cont in contaminations:
    ifo = IsolationForest(n_estimators=200, contamination=cont,
                          random_state=42, n_jobs=-1)
    ifo.fit(X_sc)
    preds = ifo.predict(X_sc)
    scores = -ifo.score_samples(X_sc)

    # Separación: diferencia de medias entre normales y anomalías
    sep = scores[preds==-1].mean() - scores[preds==1].mean()

    # Consistencia por máquina: std de la tasa de anomalía entre máquinas
    rates = [((ifo.predict(sc.transform(
                   sample[sample['machine']==m][FEAT_COLS].values)) == -1).mean())
             for m in sorted(sample['machine'].unique())]
    consistency = 1 - np.std(rates)

    cont_results.append({
        'contamination': cont,
        'anomaly_rate':  (preds==-1).mean(),
        'score_sep':     round(sep, 4),
        'consistency':   round(consistency, 4),
        'score_mean':    round(scores.mean(), 4),
    })

cont_df = pd.DataFrame(cont_results)
print(cont_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(cont_df['contamination'], cont_df['anomaly_rate'], 'o-', color='#e74c3c')
axes[0].set_title('Tasa anomalías'); axes[0].set_xlabel('contamination')
axes[1].plot(cont_df['contamination'], cont_df['score_sep'], 'o-', color='#3498db')
axes[1].set_title('Separación de scores'); axes[1].set_xlabel('contamination')
axes[2].plot(cont_df['contamination'], cont_df['consistency'], 'o-', color='#2ecc71')
axes[2].set_title('Consistencia entre máquinas'); axes[2].set_xlabel('contamination')
plt.suptitle('Isolation Forest — barrido de contamination', fontsize=12)
plt.tight_layout(); plt.show()

## 3. Optimización n_estimators y max_features

In [ ]:
# Fijar mejor contamination del barrido anterior
best_cont = cont_df.sort_values('score_sep', ascending=False).iloc[0]['contamination']
print(f'Mejor contamination: {best_cont}')

if_grid = [
    {'n_estimators': n, 'max_features': mf, 'max_samples': ms,
     'contamination': best_cont, 'random_state': 42, 'n_jobs': -1}
    for n  in [100, 200, 500]
    for mf in [0.5, 0.75, 1.0]
    for ms in [256, 'auto']
]

grid_results = []
for params in if_grid:
    ifo = IsolationForest(**params)
    ifo.fit(X_sc)
    preds  = ifo.predict(X_sc)
    scores = -ifo.score_samples(X_sc)
    sep    = scores[preds==-1].mean() - scores[preds==1].mean()
    grid_results.append({
        'n_estimators': params['n_estimators'],
        'max_features': params['max_features'],
        'max_samples':  str(params['max_samples']),
        'score_sep':    round(sep, 4),
        'anomaly_rate': round((preds==-1).mean(), 4),
    })

grid_df = pd.DataFrame(grid_results).sort_values('score_sep', ascending=False)
print("Top 10 configuraciones IF:")
print(grid_df.head(10).to_string(index=False))

## 4. Comparativa: IF vs One-Class SVM vs LOF

In [ ]:
best_if_params = grid_df.iloc[0]
# Muestra reducida para OCSVM y LOF (son lentos)
idx_sample = np.random.RandomState(42).choice(len(X_sc), size=min(10_000, len(X_sc)), replace=False)
X_small = X_sc[idx_sample]

# Isolation Forest (mejor config)
ifo_best = IsolationForest(
    n_estimators=int(best_if_params['n_estimators']),
    max_features=float(best_if_params['max_features']),
    contamination=best_cont, random_state=42, n_jobs=-1)
ifo_best.fit(X_small)
pred_if  = ifo_best.predict(X_small)
score_if = -ifo_best.score_samples(X_small)

# One-Class SVM
ocsvm = OneClassSVM(kernel='rbf', nu=best_cont, gamma='scale')
ocsvm.fit(X_small)
pred_ocsvm  = ocsvm.predict(X_small)
score_ocsvm = -ocsvm.decision_function(X_small)

# LOF (novelty=True para predict sobre nuevos datos)
lof = LocalOutlierFactor(n_neighbors=20, contamination=best_cont, novelty=True)
lof.fit(X_small)
pred_lof  = lof.predict(X_small)
score_lof = -lof.decision_function(X_small)

# Comparativa
models_comp = {
    'Isolation Forest': (pred_if, score_if),
    'One-Class SVM':    (pred_ocsvm, score_ocsvm),
    'LOF':              (pred_lof, score_lof),
}

comp_rows = []
for name, (preds, scores) in models_comp.items():
    sep = scores[preds==-1].mean() - scores[preds==1].mean() if (preds==-1).sum() > 0 else 0
    comp_rows.append({
        'Modelo': name,
        'Anomalías (%)': round((preds==-1).mean() * 100, 2),
        'Separación':    round(sep, 4),
    })

comp_df = pd.DataFrame(comp_rows)
print(comp_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, (preds, scores)) in zip(axes, models_comp.items()):
    ax.hist(scores[preds==1],  bins=60, alpha=0.6, label='Normal',   color='#3498db')
    ax.hist(scores[preds==-1], bins=60, alpha=0.6, label='Anomalía', color='#e74c3c')
    ax.set_title(name); ax.legend(fontsize=8)
plt.suptitle('Distribución de scores — comparativa de detectores', fontsize=12)
plt.tight_layout(); plt.show()

## 5. Análisis por máquina — mejor modelo

In [ ]:
# Entrenar el mejor IF sobre todos los datos y analizar por máquina
ifo_final = IsolationForest(
    n_estimators=int(best_if_params['n_estimators']),
    max_features=float(best_if_params['max_features']),
    contamination=best_cont, random_state=42, n_jobs=-1)
ifo_final.fit(X_sc)

machine_stats = []
for m in sorted(sample['machine'].unique()):
    mask = machines_all == m
    X_m  = X_sc[mask]
    pred_m  = ifo_final.predict(X_m)
    score_m = -ifo_final.score_samples(X_m)
    machine_stats.append({
        'Máquina': m, 'N': mask.sum(),
        'Anomalías (%)': round((pred_m==-1).mean()*100, 2),
        'Score medio':   round(score_m.mean(), 4),
        'Score p95':     round(np.percentile(score_m, 95), 4),
    })

mach_df = pd.DataFrame(machine_stats)
print(mach_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#e74c3c' if v > best_cont * 1.5 else '#3498db'
          for v in mach_df['Anomalías (%)']]
ax.bar(mach_df['Máquina'], mach_df['Anomalías (%)'], color=colors)
ax.axhline(best_cont * 100, color='gray', linestyle='--', label=f'Contamination={best_cont}')
ax.set_title('Tasa de anomalías por máquina')
ax.set_ylabel('Anomalías (%)')
ax.tick_params(axis='x', rotation=30)
ax.legend()
plt.tight_layout(); plt.show()

## 6. Guardar modelo SPARK afinado

In [ ]:
spark_bundle = {
    'model':        ifo_final,
    'scaler':       sc,
    'feature_cols': FEAT_COLS,
    'contamination': best_cont,
    'best_params': {
        'n_estimators': int(best_if_params['n_estimators']),
        'max_features':  float(best_if_params['max_features']),
        'contamination': best_cont,
    },
}

out_path = os.path.join(MODEL_DIR, 'spark_model.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(spark_bundle, f)

print(f'Guardado: {out_path}')
print(f'Isolation Forest — contamination={best_cont}')
print(f'n_estimators={int(best_if_params["n_estimators"])} | max_features={float(best_if_params["max_features"])}')